# 2. Produtividade IBGE (SIDRA) -- Soja no Parana

Mesma tabela SIDRA usada no pipeline de milho (5457 -- LSPA, variavel 112 = rendimento
medio, kg/ha), trocando apenas a categoria de produto na classificacao 782 (milho -> soja).

Em vez de fixar o codigo da categoria "Soja" no codigo (o que arriscaria usar um codigo
errado), ele e **descoberto dinamicamente** consultando os metadados da tabela na API do
IBGE e procurando a categoria cujo nome contem "Soja". Isso torna a busca robusta a
qualquer diferenca de codificacao entre tabelas/anos.

Periodo alvo desta calibracao: safras **2015/16 a 2024/25** (10 safras). Seguindo a
convencao do LSPA para lavouras de segunda quinzena do ano (semeadura out/nov, colheita
fev-mai do ano seguinte), a safra "Y/Y+1" e publicada pelo IBGE sob o ano civil **Y+1**
(ano de colheita) -- portanto usamos `ano` entre 2016 e 2025.


In [1]:
import os
import sys
import time

import pandas as pd
import requests
import sidrapy

sys.path.append(os.path.join(os.getcwd(), 'util'))
from util.utils_soja_pr import setup_paths_soja_pr

paths = setup_paths_soja_pr()

ANO_INICIO_COLHEITA = 2016  # safra 2015/16
ANO_FIM_COLHEITA = 2025     # safra 2024/25

df_locs = pd.read_excel(paths['COORDINATES'], sheet_name='SIDRA-ids')
df_locs['cod_municipio'] = df_locs['cod_municipio'].astype(str)
df_locs.head()


,cod_municipio,name,lat,lon,country,state,elevation_m
0,4100103,Abatiá,-23.301821,-50.338724,Brazil,Parana,609.72
1,4100202,Adrianópolis,-24.792651,-48.800739,Brazil,Parana,678.89
2,4100301,Agudos do Sul,-26.040265,-49.305534,Brazil,Parana,849.62
3,4100400,Almirante Tamandaré,-25.302081,-49.327528,Brazil,Parana,905.85
4,4100459,Altamira do Paraná,-24.824170,-52.671320,Brazil,Parana,636.07


In [2]:
# Descobrir dinamicamente o codigo da categoria "Soja" na classificacao 782 da tabela 5457,
# em vez de fixar um numero que nao temos como validar sem rodar a consulta.
METADADOS_URL = "https://servicodados.ibge.gov.br/api/v3/agregados/5457/metadados"

resp = requests.get(METADADOS_URL, timeout=30)
resp.raise_for_status()
metadados = resp.json()

categoria_soja = None
for classificacao in metadados.get('classificacoes', []):
    if classificacao['id'] == 782:
        for categoria in classificacao['categorias']:
            if 'soja' in categoria['nome'].lower():
                categoria_soja = categoria
                break

if categoria_soja is None:
    raise RuntimeError(
        "Nao foi possivel encontrar a categoria 'Soja' na classificacao 782 da tabela 5457. "
        "Verifique manualmente em " + METADADOS_URL
    )

print(f"Categoria encontrada: {categoria_soja['nome']} (id={categoria_soja['id']})")
SOJA_CLASSIFICATION_ID = str(categoria_soja['id'])


Categoria encontrada: Soja (em grão) (id=40124)


In [3]:
municipios_unicos = df_locs[['cod_municipio', 'name']].drop_duplicates()
total_municipios = len(municipios_unicos)

print(f"Buscando serie historica de rendimento de soja para {total_municipios} municipios do PR...")

all_yield_data = []
for index, row in municipios_unicos.iterrows():
    cod_municipio = row['cod_municipio']
    nome_municipio = row['name']

    print(f"({index + 1}/{total_municipios}) {nome_municipio} ({cod_municipio})...", end="")

    try:
        data = sidrapy.get_table(
            table_code="5457",
            territorial_level="6",
            ibge_territorial_code=cod_municipio,
            variable="112",
            classifications={"782": SOJA_CLASSIFICATION_ID},
            period="all"
        )

        if data is not None and len(data) > 1:
            df_municipio = data.iloc[1:].copy()
            df_municipio['cod_municipio'] = cod_municipio
            all_yield_data.append(df_municipio)
            print(" ok.")
        else:
            print(" sem dados.")

    except Exception as e:
        print(f" falha: {e}")

    time.sleep(0.5)

df_yield_raw = pd.concat(all_yield_data, ignore_index=True)
df_yield_raw.shape


Buscando serie historica de rendimento de soja para 399 municipios do PR...
(1/399) Abatiá (4100103)... ok.
(2/399) Adrianópolis (4100202)... ok.
(3/399) Agudos do Sul (4100301)... ok.
(4/399) Almirante Tamandaré (4100400)... ok.
(5/399) Altamira do Paraná (4100459)... ok.
(6/399) Altônia (4100509)... ok.
(7/399) Alto Paraná (4100608)... ok.
(8/399) Alto Piquiri (4100707)... ok.
(9/399) Alvorada do Sul (4100806)... ok.
(10/399) Amaporã (4100905)... ok.
(11/399) Ampére (4101002)... ok.
(12/399) Anahy (4101051)... ok.
(13/399) Andirá (4101101)... ok.
(14/399) Ângulo (4101150)... ok.
(15/399) Antonina (4101200)... ok.
(16/399) Antônio Olinto (4101309)... ok.
(17/399) Apucarana (4101408)... ok.
(18/399) Arapongas (4101507)... ok.
(19/399) Arapoti (4101606)... ok.
(20/399) Arapuã (4101655)... ok.
(21/399) Araruna (4101705)... ok.
(22/399) Araucária (4101804)... ok.
(23/399) Ariranha do Ivaí (4101853)... ok.
(24/399) Assaí (4101903)... ok.
(25/399) Assis Chateaubriand (4102000)... ok.
(26/39

(20349, 14)

In [4]:
df_yield = df_yield_raw.rename(columns={
    'D1C': 'cod_municipio_api',
    'D1N': 'municipio_nome',
    'D2N': 'ano',
    'V': 'yield (kg/ha)',
})[['cod_municipio', 'municipio_nome', 'ano', 'yield (kg/ha)']]

df_yield['ano'] = pd.to_numeric(df_yield['ano'], errors='coerce')
df_yield['yield (kg/ha)'] = pd.to_numeric(df_yield['yield (kg/ha)'], errors='coerce')
df_yield.dropna(inplace=True)
df_yield['ano'] = df_yield['ano'].astype(int)

# Restringe ao periodo de calibracao (safras 2015/16 a 2024/25, rotuladas pelo ano de colheita)
df_yield = df_yield[(df_yield['ano'] >= ANO_INICIO_COLHEITA) & (df_yield['ano'] <= ANO_FIM_COLHEITA)]

df_yield.head()


,cod_municipio,municipio_nome,ano,yield (kg/ha)
42,4100103,Abatiá (PR),2016,2760.0
43,4100103,Abatiá (PR),2017,3480.0
44,4100103,Abatiá (PR),2018,2640.0
45,4100103,Abatiá (PR),2019,3480.0
46,4100103,Abatiá (PR),2020,3480.0


In [5]:
df_yield_final = pd.merge(
    df_locs[['lat', 'lon', 'country', 'state', 'cod_municipio', 'name', 'elevation_m']],
    df_yield[['cod_municipio', 'ano', 'yield (kg/ha)']],
    on='cod_municipio',
    how='inner'
).rename(columns={'name': 'county'})

print(f"Municipios com pelo menos 1 ano de dados de soja no periodo: {df_yield_final['cod_municipio'].nunique()} / {len(df_locs)}")

with pd.ExcelWriter(paths['COORDINATES'], mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    df_yield_final.to_excel(writer, sheet_name='yield_PR_soja', index=False)

df_yield_final.to_csv(os.path.join(paths['DATA'], 'yield_soja_pr.csv'), index=False)

print(f"Dados de produtividade salvos em {paths['COORDINATES']} (aba yield_PR_soja)")
df_yield_final.head()


Municipios com pelo menos 1 ano de dados de soja no periodo: 392 / 399
Dados de produtividade salvos em d:\_py\AgroIA_prod\inputs\data\soja_pr\coordinates_pr.xlsx (aba yield_PR_soja)


,lat,lon,country,state,cod_municipio,county,elevation_m,ano,yield (kg/ha)
0,-23.301821,-50.338724,Brazil,Parana,4100103,Abatiá,609.72,2016,2760.0
1,-23.301821,-50.338724,Brazil,Parana,4100103,Abatiá,609.72,2017,3480.0
2,-23.301821,-50.338724,Brazil,Parana,4100103,Abatiá,609.72,2018,2640.0
3,-23.301821,-50.338724,Brazil,Parana,4100103,Abatiá,609.72,2019,3480.0
4,-23.301821,-50.338724,Brazil,Parana,4100103,Abatiá,609.72,2020,3480.0
